In [ ]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [ ]:
import gzip
import json
import sqlite3
from pathlib import Path

from tqdm import tqdm

In [ ]:
import pandas as pd

In [ ]:
import synth_extract.utils.sql_helpers as sql
# importlib.reload(sql)

#### S2ORC DB

scripts to create the sql db for S2ORC

In [ ]:
# s2orc_path = "../data/s2orc/s2orc_filtered_polymer.jsonl.gz"
# db_path = "../data/s2orc_filtered_polymer.db"

# BATCH_SIZE = 10_000

# conn = sqlite3.connect(db_path)
# cursor = conn.cursor()

# cursor.execute("""
# CREATE TABLE IF NOT EXISTS papers (
#     corpusid INTEGER PRIMARY KEY,
#     doi TEXT,
#     title TEXT,
#     authors TEXT,
#     abstract TEXT
# );
# """)


# insert_query = """
# INSERT OR REPLACE INTO papers (
#     corpusid,
#     doi,
#     title,
#     authors,
#     abstract
# )
# VALUES (?, ?, ?, ?, ?);
# """

# batch = []
# inserted = 0
# skipped = 0

# with gzip.open(s2orc_path, "rt", encoding="utf-8") as file:
#     for line_number, line in enumerate(
#         tqdm(file, desc="Converting JSONL to SQLite"),
#         start=1,
#     ):
#         line = line.strip()

#         if not line:
#             continue

#         try:
#             record = json.loads(line)

#             corpusid = record.get("corpusid")

#             # A primary key cannot be missing
#             if corpusid is None:
#                 skipped += 1
#                 continue

#             open_access_info = record.get("openaccessinfo") or {}
#             external_ids = open_access_info.get("externalids") or {}

#             # Also check top-level externalids as a fallback
#             if not external_ids:
#                 external_ids = record.get("externalids") or {}

#             doi = external_ids.get("DOI")
#             title = record.get("title")
#             authors = record.get("authors") or []
#             abstract = record.get("abstract")

#             # Store the list of authors as JSON text
#             authors_json = json.dumps(
#                 authors,
#                 ensure_ascii=False,
#             )

#             batch.append(
#                 (
#                     corpusid,
#                     doi,
#                     title,
#                     authors_json,
#                     abstract,
#                 )
#             )

#             if len(batch) >= BATCH_SIZE:
#                 cursor.executemany(insert_query, batch)
#                 conn.commit()

#                 inserted += len(batch)
#                 batch.clear()

#         except json.JSONDecodeError as error:
#             skipped += 1
#             print(
#                 f"\nInvalid JSON on line {line_number}: {error}"
#             )

#         except Exception as error:
#             skipped += 1
#             print(
#                 f"\nError processing line {line_number}: {error}"
#             )

# # Insert remaining records
# if batch:
#     cursor.executemany(insert_query, batch)
#     conn.commit()
#     inserted += len(batch)

# conn.close()

# print(f"Database created: {Path(db_path).resolve()}")
# print(f"Records processed: {inserted:,}")
# print(f"Records skipped: {skipped:,}")

#### Normalize DBs

Deduplicate, remove invalid entries and copy to a fresh db

In [ ]:
import importlib

In [ ]:
import synth_extract.utils.sql_helpers as sql
importlib.reload(sql)

In [ ]:
db_path = "../data/europepmc.db"

In [ ]:
sql.list_tables(db_path)

In [ ]:
sql.get_table_schema(db_path, "papers")

In [ ]:
df = sql.load_table(db_path, "papers")

In [ ]:
df.head()

In [ ]:
# df["abstract_source"].value_counts()

In [ ]:
# Columns that are required
required_cols = ["pmcid", "doi", "title", "abstract"]

# Statistics
stats = {
    "total_rows": len(df),
}

for col in required_cols:
    stats[f"empty_{col}"] = (
        df[col].isna() | (df[col].astype(str).str.strip() == "")
    ).sum()

# Rows where any required field is missing
missing_mask = pd.Series(False, index=df.index)

for col in required_cols:
    missing_mask |= (
        df[col].isna() | (df[col].astype(str).str.strip() == "")
    )

stats["rows_with_missing_required_fields"] = missing_mask.sum()
stats["rows_passing"] = (~missing_mask).sum()

# Print nicely
for k, v in stats.items():
    print(f"{k:35s}: {v:,}")

In [ ]:
def quote_identifier(name: str) -> str:
    """Safely quote a SQLite table or column identifier."""
    return '"' + name.replace('"', '""') + '"'


def copy_table(
    db_path: str,
    source_table: str,
    output_table: str = "papers_cleaned",
    overwrite: bool = False,
) -> None:
    database_path = Path(db_path).expanduser().resolve()

    if not database_path.exists():
        raise FileNotFoundError(
            f"Database not found: {database_path}"
        )

    if source_table == output_table:
        raise ValueError(
            "Source table and output table must have different names."
        )

    source_sql = quote_identifier(source_table)
    output_sql = quote_identifier(output_table)

    with sqlite3.connect(database_path) as conn:
        # Check that source table exists
        source_exists = conn.execute(
            """
            SELECT 1
            FROM sqlite_master
            WHERE type = 'table' AND name = ?
            """,
            (source_table,),
        ).fetchone()

        if source_exists is None:
            raise ValueError(
                f"Source table {source_table!r} does not exist."
            )

        # Check whether output table already exists
        output_exists = conn.execute(
            """
            SELECT 1
            FROM sqlite_master
            WHERE type = 'table' AND name = ?
            """,
            (output_table,),
        ).fetchone()

        if output_exists:
            if overwrite:
                conn.execute(f"DROP TABLE {output_sql}")
            else:
                raise ValueError(
                    f"Table {output_table!r} already exists. "
                    "Use overwrite=True to replace it."
                )

        source_count = conn.execute(
            f"SELECT COUNT(*) FROM {source_sql}"
        ).fetchone()[0]

        # Copy every column and every row
        conn.execute(
            f"""
            CREATE TABLE {output_sql} AS
            SELECT *
            FROM {source_sql}
            """
        )

        copied_count = conn.execute(
            f"SELECT COUNT(*) FROM {output_sql}"
        ).fetchone()[0]

        conn.commit()

    print(f"Database       : {database_path}")
    print(f"Source table   : {source_table}")
    print(f"Output table   : {output_table}")
    print(f"Source rows    : {source_count:,}")
    print(f"Copied rows    : {copied_count:,}")


def create_cleaned_table(
    db_path: str,
    source_table: str,
    columns: list[str],
    output_table: str = "papers_cleaned",
    overwrite: bool = False,
) -> None:
    """
    Create a cleaned table in the same SQLite database.

    Only rows with non-empty DOI, title, and abstract are copied.

    Parameters
    ----------
    db_path
        Path to the SQLite database.
    source_table
        Name of the existing source table.
    columns
        Columns to copy into the cleaned table.
    output_table
        Name of the new cleaned table.
    overwrite
        Replace the output table if it already exists.
    """
    database_path = Path(db_path).expanduser().resolve()

    if not database_path.exists():
        raise FileNotFoundError(
            f"Database not found: {database_path}"
        )

    if not columns:
        raise ValueError("The columns list cannot be empty.")

    if source_table == output_table:
        raise ValueError(
            "The source table and output table must have different names."
        )

    required_filter_columns = {"doi", "title", "abstract"}

    source_table_sql = quote_identifier(source_table)
    output_table_sql = quote_identifier(output_table)

    with sqlite3.connect(database_path) as conn:
        # Confirm that the source table exists
        table_exists = conn.execute(
            """
            SELECT 1
            FROM sqlite_master
            WHERE type = 'table' AND name = ?
            """,
            (source_table,),
        ).fetchone()

        if table_exists is None:
            raise ValueError(
                f"Table {source_table!r} does not exist in "
                f"{database_path}"
            )

        # Read columns from the source-table schema
        source_schema = conn.execute(
            f"PRAGMA table_info({source_table_sql})"
        ).fetchall()

        source_columns = {row[1] for row in source_schema}

        # Check the requested output columns
        missing_requested = set(columns) - source_columns

        if missing_requested:
            raise ValueError(
                "The following requested columns are not present in "
                f"{source_table!r}: {sorted(missing_requested)}"
            )

        # Check columns required for filtering
        missing_filter_columns = (
            required_filter_columns - source_columns
        )

        if missing_filter_columns:
            raise ValueError(
                "The source table does not contain the columns required "
                f"for filtering: {sorted(missing_filter_columns)}"
            )

        # Check whether the destination table already exists
        output_exists = conn.execute(
            """
            SELECT 1
            FROM sqlite_master
            WHERE type = 'table' AND name = ?
            """,
            (output_table,),
        ).fetchone()

        if output_exists:
            if overwrite:
                conn.execute(
                    f"DROP TABLE {output_table_sql}"
                )
            else:
                raise ValueError(
                    f"Table {output_table!r} already exists. "
                    "Use overwrite=True to replace it."
                )

        selected_columns_sql = ", ".join(
            quote_identifier(c) for c in columns
        )

        total_rows = conn.execute(
            f"SELECT COUNT(*) FROM {source_table_sql}"
        ).fetchone()[0]

        # CREATE TABLE AS SELECT creates and populates the table
        conn.execute(
            f"""
            CREATE TABLE {output_table_sql} AS
            SELECT
                {selected_columns_sql}
            FROM {source_table_sql}
            WHERE
                doi IS NOT NULL
                AND TRIM(doi) <> ''
                AND title IS NOT NULL
                AND TRIM(title) <> ''
                AND abstract IS NOT NULL
                AND TRIM(abstract) <> ''
            """
        )

        copied_rows = conn.execute(
            f"SELECT COUNT(*) FROM {output_table_sql}"
        ).fetchone()[0]

        conn.commit()

    print(f"Database       : {database_path}")
    print(f"Source table   : {source_table}")
    print(f"Output table   : {output_table}")
    print(f"Columns copied : {', '.join(columns)}")
    print(f"Original rows  : {total_rows:,}")
    print(f"Copied rows    : {copied_rows:,}")
    print(f"Rejected rows  : {total_rows - copied_rows:,}")

In [ ]:
sql.list_columns(db_path, "papers")

In [ ]:
# columns = ['pmcid',
#  'pmid',
#  'doi',
#  'title',
#  'journal',
#  'abstract']
 
# create_cleaned_table(db_path=db_path, source_table="papers",
#                      columns=columns)

#### DOI Duplicates Check

In [ ]:
sources = [
    {
        "source": "wiley",
        "db_path": "../data/scopus_wiley.db",
        "table": "papers_dup",
    },
    {
        "source": "springer_nature",
        "db_path": "../data/scopus_springer_nature.db",
        "table": "papers_dup",
    },
    {
        "source": "elsevier",
        "db_path": "../data/scopus_elsevier.db",
        "table": "papers_dup",
    },
    {
        "source": "s2orc",
        "db_path": "../data/s2orc.db",
        "table": "papers_dup",
    },
    {
        "source": "europepmc",
        "db_path": "../data/europepmc.db",
        "table": "papers_dup",
    },
]

In [ ]:
source_dfs = []

for item in sources:
    db_path = Path(item["db_path"]).expanduser().resolve()
    table_name = item["table"]
    source_name = item["source"]

    if not db_path.exists():
        raise FileNotFoundError(
            f"Database not found for {source_name}: {db_path}"
        )

    with sqlite3.connect(db_path) as conn:
        # Check that the table exists
        table_exists = conn.execute(
            """
            SELECT 1
            FROM sqlite_master
            WHERE type = 'table' AND name = ?
            """,
            (table_name,),
        ).fetchone()

        if table_exists is None:
            raise ValueError(
                f"Table {table_name!r} not found in {db_path}"
            )

        # Check that the DOI column exists
        schema = conn.execute(
            f'PRAGMA table_info("{table_name}")'
        ).fetchall()

        columns = {row[1] for row in schema}

        if "doi" not in columns:
            raise ValueError(
                f"Table {table_name!r} in {db_path} "
                "does not contain a 'doi' column."
            )

        source_df = pd.read_sql_query(
            f'SELECT doi FROM "{table_name}"',
            conn,
        )

    source_df["source"] = source_name
    source_dfs.append(source_df)

    print(
        f"{source_name:20s}: "
        f"{len(source_df):,} rows loaded"
    )

In [ ]:
df_all = pd.concat(
    source_dfs,
    ignore_index=True,
)

print(f"Combined rows: {len(df_all):,}")

In [ ]:
import re

def normalize_doi(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if not value:
        return pd.NA

    # Remove common DOI prefixes
    value = re.sub(
        r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Remove surrounding whitespace and common trailing punctuation
    value = value.strip().rstrip(".,;")

    return value if value else pd.NA

In [ ]:
df_all["doi_normalized"] = df_all["doi"].apply(
    normalize_doi
)

In [ ]:
source_statistics = (
    df_all
    .groupby("source")
    .agg(
        total_rows=("doi", "size"),
        missing_doi_rows=(
            "doi_normalized",
            lambda x: x.isna().sum(),
        ),
        valid_doi_rows=(
            "doi_normalized",
            lambda x: x.notna().sum(),
        ),
        unique_valid_dois=(
            "doi_normalized",
            lambda x: x.dropna().nunique(),
        ),
    )
    .reset_index()
)

source_statistics

In [ ]:
df_doi_sources = (
    df_all
    .dropna(subset=["doi_normalized"])
    [["doi_normalized", "source"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Unique DOI/source pairs: {len(df_doi_sources):,}")

In [ ]:
total_unique_dois = (
    df_doi_sources["doi_normalized"].nunique()
)

print(f"Total unique DOIs: {total_unique_dois:,}")

In [ ]:
doi_source_counts = (
    df_doi_sources
    .groupby("doi_normalized")
    .agg(
        source_count=("source", "nunique"),
        sources=(
            "source",
            lambda x: sorted(x.unique()),
        ),
    )
    .reset_index()
)

print(doi_source_counts.head())

In [ ]:
dois_in_one_source = (
    doi_source_counts["source_count"] == 1
).sum()

print(
    f"DOIs in exactly one source: "
    f"{dois_in_one_source:,}"
)

In [ ]:
dois_in_multiple_sources = (
    doi_source_counts["source_count"] > 1
).sum()

print(
    f"DOIs in multiple sources: "
    f"{dois_in_multiple_sources:,}"
)

In [ ]:
doi_source_distribution = (
    doi_source_counts["source_count"]
    .value_counts()
    .sort_index()
    .rename_axis("source_count")
    .reset_index(name="doi_count")
)

print(doi_source_distribution)

In [ ]:
pairwise_records = []

source_names = sorted(
    df_doi_sources["source"].unique()
)

for i, source_1 in enumerate(source_names):
    dois_1 = set(
        df_doi_sources.loc[
            df_doi_sources["source"] == source_1,
            "doi_normalized",
        ]
    )

    for source_2 in source_names[i + 1:]:
        dois_2 = set(
            df_doi_sources.loc[
                df_doi_sources["source"] == source_2,
                "doi_normalized",
            ]
        )

        pairwise_records.append(
            {
                "source_1": source_1,
                "source_2": source_2,
                "shared_dois": len(dois_1 & dois_2),
            }
        )

pairwise_overlap = (
    pd.DataFrame(pairwise_records)
    .sort_values("shared_dois", ascending=False)
    .reset_index(drop=True)
)

# pairwise_overlap = (
#     pd.DataFrame(pairwise_records)
#     .sort_values(
#         ["source_1", "source_2"],
#         ignore_index=True,
#     )
# )

print(pairwise_overlap)

at this point we need clear investigation on dupicate dois in a single source

#### S2ORC

start with s2orc and deduplicate this

In [ ]:
df = sql.load_table("../data/s2orc.db", "papers_cleaned")

In [ ]:
def normalize_doi(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if not value:
        return pd.NA

    # Remove common DOI prefixes
    value = re.sub(
        r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Remove surrounding whitespace and common trailing punctuation
    value = value.strip().rstrip(".,;")

    return value if value else pd.NA

In [ ]:
df["doi_normalized"] = df["doi"].apply(normalize_doi)

In [ ]:
total_rows = len(df)

unique_dois = df["doi_normalized"].nunique(dropna=True)

duplicate_rows = total_rows - unique_dois

duplicate_dois = (
    df["doi_normalized"]
    .dropna()
    .value_counts()
    .loc[lambda x: x > 1]
)

print(f"Total rows           : {total_rows:,}")
print(f"Unique DOIs          : {unique_dois:,}")
print(f"Duplicate rows       : {duplicate_rows:,}")
print(f"Duplicate DOI values : {len(duplicate_dois):,}")

In [ ]:
dup_distribution = (
    duplicate_dois
    .value_counts()
    .sort_index()
    .rename_axis("times_appearing")
    .reset_index(name="number_of_dois")
)

print(dup_distribution)

In [ ]:
doi = duplicate_dois.index[0]

example = (
    df[df["doi_normalized"] == doi]
    .sort_values("corpusid")
)

example

In [ ]:
# DOI counts
doi_counts = df["doi_normalized"].value_counts()

# Keep only rows whose DOI appears more than once
duplicate_df = df[df["doi_normalized"].isin(doi_counts[doi_counts > 1].index)].copy()

# Count distinct titles per duplicated DOI
title_comparison = (
    duplicate_df
    .groupby("doi_normalized")
    .agg(
        row_count=("doi_normalized", "size"),
        unique_title_count=("title", "nunique"),
        titles=("title", lambda x: list(pd.unique(x))),
    )
    .reset_index()
)

same_title_count = (title_comparison["unique_title_count"] == 1).sum()
different_title_count = (title_comparison["unique_title_count"] > 1).sum()

print(f"Duplicate DOIs with the same title      : {same_title_count:,}")
print(f"Duplicate DOIs with different titles   : {different_title_count:,}")
print(f"Total duplicated DOI groups            : {len(title_comparison):,}")

In [ ]:
different_titles = title_comparison[
    title_comparison["unique_title_count"] > 1
].copy()

display(different_titles)

In [ ]:
different_title_dois = title_comparison.loc[
    title_comparison["unique_title_count"] > 1,
    "doi_normalized",
]

different_title_rows = (
    duplicate_df.loc[
        duplicate_df["doi_normalized"].isin(different_title_dois),
        ["doi_normalized", "corpusid", "title", "authors"],
    ]
    .sort_values(["doi_normalized", "corpusid"])
)

In [ ]:
c = 0
for doi, group in different_title_rows.groupby("doi_normalized"):
    c+=1
    print("=" * 100)
    print(f"Number: {c}")
    print(f"DOI: {doi}")
    print(f"Number of records: {len(group)}")
    print()

    for _, row in group.iterrows():
        print(f"Corpus ID: {row['corpusid']}")
        print(f"Title    : {row['title']}")
        print(f"Authors  : {row['authors']}")
        print("-" * 100)

    print()

In [ ]:
# corpus id 8950287 change doi to 10.1109/jsen.2005.847935
# corpus id 116154488 change doi to 10.1016/s0167-2991(05)80497-5 
# corpus id 6067032 chnage doi to 10.1016/j.jcp.2010.12.028 
# corpus id 240357650 change doi to 10.1016/j.bpj.2021.11.2552 

In [ ]:
corpusid_to_remove = [243832963, 259223711, 215615534, 253843521, 240154179, 254546603, 256091032, 257116500, 219333773, 232407699, 119196911, 236885221, 233201112, 244131605,
                      248154740, 248079471, 119206637, 55533958, 222133937, 219212149, 219208757, 219212895, 219212178, 219210460, 219209683, 219213004, 219216795, 219199429,
                      219213385, 219202821, 219197239, 219200942, 53956338, 219227052, 219223513, 219236605, 219215367, 203656342, 219232484, 219225334, 219234767, 85888555,
                      219229965, 219225587, 219238524, 212631896, 219208921, 220287727, 220287727, 220294008, 115918550, 219236130, 85799060, 219232943, 12363943, 231634954,
                      225052047, 119105714, 16250154, 6222232, 135675746, 135838168, 24594306, 216647290, 220940864, 90411951, 19354463, 214725380, 220922765, 220885260, 221662281, 221609670,
                      221823705, 222799070, 237108418, 227060927, 227922456, 233246024, 244492126, 245148711, 245148488, 89968430, 91473062, 214725140, 201193851, 214722078, 203878120,
                      119267420, 221222260, 221637344, 240846878, 54646670, 82156266, 5293592, 255783203, 255781041, 17417448, 255808010, 7678960, 3029757, 2311253, 13901980, 6476945,
                      255844335, 255955956, 255978017, 256332577, 256208136, 255816797, 255811519, 11817372, 2840670, 255983087, 54590743, 7827951, 55834377, 14306246, 218647290, 55612585,
                      67826255, 203995291, 962046, 46947618, 46949206, 8440815, 54552629, 138461539, 213206893, 226203432, 185155064, 10492106, 2274212, 8534614, 208245870, 90667934, 231553994,
                      222131030, 119106612, 199575121, 191186013, 237538869, 53398098, 227948043, 234743312, 232198020, 88714230, 218621340, 20758480, 219319681, 99380799, 236090409, 247382974,
                      216123178, 242205993, 21823772, 219219387, 225053497, 231633244]

corpusid_to_remove.extend([1342682, 54567999, 219063615, 221637129, 24034960, 54526132, 225040267, 250670157]) # not relevent papers found from abstract check

## THIS LIST CONTAINS NOT JUST DUPLICATES BUT SOME CORPUS IDS THAT I DECIDED TO REMOVE

In [ ]:
## same title ones i need to check abstract ....

import re
import unicodedata

import pandas as pd
from rapidfuzz import fuzz


# Adjust this threshold after inspecting examples
abstract_threshold = 90


def normalize_abstract(value):
    """Normalize superficial formatting differences in abstracts."""
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKC", str(value))
    value = value.lower().strip()

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    # Remove spaces before punctuation
    value = re.sub(r"\s+([.,;:!?])", r"\1", value)

    return value


def compare_abstracts(abstracts, threshold=90):
    """
    Compare every abstract pair within one DOI group.

    A group passes only when every pair meets the threshold.
    """
    normalized = [
        normalize_abstract(abstract)
        for abstract in abstracts
    ]

    # Retain missing abstracts so a missing-vs-filled pair is detected
    normalized = list(dict.fromkeys(normalized))

    if len(normalized) <= 1:
        return {
            "abstracts_similar": True,
            "min_similarity": 100.0,
            "mean_similarity": 100.0,
            "max_similarity": 100.0,
            "pair_count": 0,
        }

    scores = []

    for i in range(len(normalized)):
        for j in range(i + 1, len(normalized)):
            abstract_1 = normalized[i]
            abstract_2 = normalized[j]

            if not abstract_1 and not abstract_2:
                score = 100.0
            elif not abstract_1 or not abstract_2:
                score = 0.0
            else:
                score = fuzz.ratio(abstract_1, abstract_2)

            scores.append(score)

    return {
        "abstracts_similar": min(scores) >= threshold,
        "min_similarity": min(scores),
        "mean_similarity": sum(scores) / len(scores),
        "max_similarity": max(scores),
        "pair_count": len(scores),
    }

same_title_dois = title_comparison.loc[
    title_comparison["unique_title_count"] == 1,
    "doi_normalized",
]

same_title_df = duplicate_df[
    duplicate_df["doi_normalized"].isin(same_title_dois)
].copy()

abstract_comparison = (
    same_title_df
    .groupby("doi_normalized")
    .agg(
        row_count=("doi_normalized", "size"),
        corpusids=("corpusid", lambda x: list(x)),
        title=("title", "first"),
        abstracts=("abstract", lambda x: list(x)),
    )
    .reset_index()
)

comparison_results = abstract_comparison["abstracts"].apply(
    lambda abstracts: compare_abstracts(
        abstracts,
        threshold=abstract_threshold,
    )
)

comparison_results = pd.DataFrame(
    comparison_results.tolist(),
    index=abstract_comparison.index,
)

abstract_comparison = pd.concat(
    [abstract_comparison, comparison_results],
    axis=1,
)

similar_count = abstract_comparison["abstracts_similar"].sum()
different_count = (~abstract_comparison["abstracts_similar"]).sum()

print(f"Similarity threshold                    : {abstract_threshold}")
print(f"Same-title DOI groups                   : {len(abstract_comparison):,}")
print(f"Groups with similar abstracts           : {similar_count:,}")
print(f"Groups with different abstracts         : {different_count:,}")

In [ ]:
suspicious_abstracts = (
    abstract_comparison
    .loc[
        ~abstract_comparison["abstracts_similar"],
        [
            "doi_normalized",
            "row_count",
            "corpusids",
            "title",
            "min_similarity",
            "mean_similarity",
            "abstracts",
        ],
    ]
    .sort_values("min_similarity")
    .reset_index(drop=True)
)

for _, row in suspicious_abstracts.head(10).iterrows():
    print("=" * 120)
    print(f"DOI: {row['doi_normalized']}")
    print(f"Corpus IDs: {row['corpusids']}")
    print(f"Title: {row['title']}")
    print(f"Minimum similarity: {row['min_similarity']:.2f}")
    print()

    for i, abstract in enumerate(row["abstracts"], start=1):
        print(f"ABSTRACT {i}")
        print(abstract)
        print("-" * 120)

In [ ]:
## create new deduplicated table

# Start from a separate copy.
df_cleaned = df.copy(deep=True)

doi_updates = {
    8950287: "10.1109/jsen.2005.847935",
    116154488: "10.1016/s0167-2991(05)80497-5",
    6067032: "10.1016/j.jcp.2010.12.028",
    240357650: "10.1016/j.bpj.2021.11.2552",
}

# Update DOI values.
update_mask = df_cleaned["corpusid"].isin(doi_updates)

df_cleaned.loc[update_mask, "doi"] = (
    df_cleaned.loc[update_mask, "corpusid"]
    .map(doi_updates)
)

# Recalculate normalized DOI using your function.
df_cleaned.loc[update_mask, "doi_normalized"] = (
    df_cleaned.loc[update_mask, "doi"]
    .map(normalize_doi)
)

# Remove unwanted corpus IDs.
df_cleaned = (
    df_cleaned.loc[
        ~df_cleaned["corpusid"].isin(corpusid_to_remove)
    ]
    .copy()
    .reset_index(drop=True)
)

In [ ]:
# Remove duplicate normalized DOIs, keeping the first occurrence.
df_cleaned = (
    df_cleaned
    .drop_duplicates(subset="doi_normalized", keep="first")
    .reset_index(drop=True)
)

In [ ]:
df_cleaned

In [ ]:
# # Replace doi with the normalized DOI before writing.
# df_to_sql = (
#     df_cleaned
#     .copy()
#     .assign(doi=lambda x: x["doi_normalized"])
#     [["corpusid", "doi", "title", "authors", "abstract"]]
# )

# with sqlite3.connect('../data/s2orc.db') as conn:
#     conn.execute("DROP TABLE IF EXISTS papers_dup")

#     conn.execute("""
#         CREATE TABLE papers_dup (
#             corpusid INTEGER PRIMARY KEY,
#             doi TEXT,
#             title TEXT,
#             authors TEXT,
#             abstract TEXT
#         )
#     """)

#     df_to_sql.to_sql(
#         "papers_dup",
#         conn,
#         if_exists="append",
#         index=False,
#         chunksize=500,
#     )

# print(f"Wrote {len(df_to_sql):,} rows to papers_dup.")

In [ ]:
sql.list_tables("../data/s2orc.db")

#### Springer Nature

Nothing major needs to be done. we are normalizing dois as adding

In [ ]:
sql.list_tables("../data/scopus_springer_nature.db")

In [ ]:
sql.get_table_schema("../data/scopus_springer_nature.db", "papers_cleaned")

In [ ]:
# db_path = "../data/scopus_springer_nature.db"

# with sqlite3.connect(db_path) as conn:
#     df_dup = pd.read_sql("SELECT * FROM papers_cleaned", conn)

# df_dup["doi"] = df_dup["doi"].map(normalize_doi)

# with sqlite3.connect(db_path) as conn:
#     conn.execute("DROP TABLE IF EXISTS papers_dup")
#     conn.execute("""
#         CREATE TABLE papers_dup (
#             eid TEXT PRIMARY KEY,
#             doi TEXT,
#             title TEXT,
#             journal TEXT,
#             publisher TEXT,
#             abstract TEXT
#         )
#     """)

#     df_dup.to_sql(
#         "papers_dup",
#         conn,
#         if_exists="append",
#         index=False,
#         chunksize=500,
#     )

In [ ]:
sql.get_table_schema("../data/scopus_springer_nature.db", "papers_dup")

#### Elsevier

Here we have some duplicates

In [ ]:
df = sql.load_table("../data/scopus_elsevier.db", "papers_cleaned")

In [ ]:
df.head()

In [ ]:
def normalize_doi(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if not value:
        return pd.NA

    # Remove common DOI prefixes
    value = re.sub(
        r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Remove surrounding whitespace and common trailing punctuation
    value = value.strip().rstrip(".,;")

    return value if value else pd.NA

In [ ]:
df["doi"] = df["doi"].apply(normalize_doi)

In [ ]:
total_rows = len(df)

unique_dois = df["doi"].nunique(dropna=True)

duplicate_rows = total_rows - unique_dois

duplicate_dois = (
    df["doi"]
    .dropna()
    .value_counts()
    .loc[lambda x: x > 1]
)

print(f"Total rows           : {total_rows:,}")
print(f"Unique DOIs          : {unique_dois:,}")
print(f"Duplicate rows       : {duplicate_rows:,}")
print(f"Duplicate DOI values : {len(duplicate_dois):,}")

In [ ]:
dup_distribution = (
    duplicate_dois
    .value_counts()
    .sort_index()
    .rename_axis("times_appearing")
    .reset_index(name="number_of_dois")
)

print(dup_distribution)

In [ ]:
doi = duplicate_dois.index[0]

example = (
    df[df["doi"] == doi]
    .sort_values("eid")
)

example

In [ ]:
def normalize_title(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if not value:
        return pd.NA

    # Remove punctuation and normalize whitespace.
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value if value else pd.NA


df_check = df.copy()
df_check["title_normalized"] = df_check["title"].map(normalize_title)

# DOI counts
doi_counts = df_check["doi"].value_counts()

# Keep only rows whose DOI appears more than once
duplicate_df = df_check[
    df_check["doi"].isin(doi_counts[doi_counts > 1].index)
].copy()

# Count distinct normalized titles per duplicated DOI
title_comparison = (
    duplicate_df
    .groupby("doi")
    .agg(
        row_count=("doi", "size"),
        unique_title_count=("title_normalized", "nunique"),
        titles=("title", lambda x: list(pd.unique(x))),
        normalized_titles=(
            "title_normalized",
            lambda x: list(pd.unique(x.dropna())),
        ),
    )
    .reset_index()
)

same_title_count = (
    title_comparison["unique_title_count"] == 1
).sum()

different_title_count = (
    title_comparison["unique_title_count"] > 1
).sum()

print(
    f"Duplicate DOIs with the same normalized title    : "
    f"{same_title_count:,}"
)
print(
    f"Duplicate DOIs with different normalized titles : "
    f"{different_title_count:,}"
)
print(
    f"Total duplicated DOI groups                      : "
    f"{len(title_comparison):,}"
)

In [ ]:
different_titles = title_comparison[
    title_comparison["unique_title_count"] > 1
].copy()

display(different_titles)

In [ ]:
different_title_dois = title_comparison.loc[
    title_comparison["unique_title_count"] > 1,
    "doi",
]

different_title_rows = (
    duplicate_df.loc[
        duplicate_df["doi"].isin(different_title_dois),
        ["doi", "eid", "title", "abstract"],
    ]
    .sort_values(["doi", "eid"])
)

c = 0
for doi, group in different_title_rows.groupby("doi"):
    c+=1
    print("=" * 100)
    print(f"Number: {c}")
    print(f"DOI: {doi}")
    print(f"Number of records: {len(group)}")
    print()

    for _, row in group.iterrows():
        print(f"eID.     : {row['eid']}")
        print(f"Title    : {row['title']}")
        print(f"Abstract  : {row['abstract']}")
        print("-" * 100)

    print()

In [ ]:
from itertools import combinations

import pandas as pd
from rapidfuzz import fuzz


FUZZY_THRESHOLD = 90


def prepare_text(value):
    if pd.isna(value):
        return ""

    return " ".join(str(value).lower().split())


different_title_dois = title_comparison.loc[
    title_comparison["unique_title_count"] > 1,
    "doi",
]

different_title_rows = (
    duplicate_df.loc[
        duplicate_df["doi"].isin(different_title_dois),
        ["doi", "eid", "title", "abstract"],
    ]
    .sort_values(["doi", "eid"])
    .copy()
)

comparison_results = []

for doi, group in different_title_rows.groupby("doi"):
    records = group.to_dict("records")
    pair_scores = []

    for record_1, record_2 in combinations(records, 2):
        abstract_score = fuzz.ratio(
            prepare_text(record_1["abstract"]),
            prepare_text(record_2["abstract"]),
        )

        pair_scores.append(
            {
                "eid_1": record_1["eid"],
                "eid_2": record_2["eid"],
                "abstract_score": abstract_score,
            }
        )

    minimum_abstract_score = min(
        pair["abstract_score"] for pair in pair_scores
    )

    comparison_results.append(
        {
            "doi": doi,
            "record_count": len(group),
            "minimum_abstract_score": minimum_abstract_score,
            "passed": minimum_abstract_score >= FUZZY_THRESHOLD,
            "pair_scores": pair_scores,
        }
    )

abstract_comparison = pd.DataFrame(comparison_results)

total_groups = len(abstract_comparison)
passed_groups = abstract_comparison["passed"].sum()
failed_groups = total_groups - passed_groups

print(f"DOI groups checked                 : {total_groups:,}")
print(f"Groups passing abstract similarity : {passed_groups:,}")
print(f"Groups failing abstract similarity : {failed_groups:,}")

failed_groups_df = (
    abstract_comparison.loc[~abstract_comparison["passed"]]
    .sort_values(
        ["minimum_abstract_score", "doi"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

c = 0

for _, result in failed_groups_df.iterrows():
    doi = result["doi"]

    group = different_title_rows.loc[
        different_title_rows["doi"] == doi
    ]

    c += 1

    print("=" * 100)
    print(f"Number: {c}")
    print(f"DOI: {doi}")
    print(f"Number of records: {len(group)}")
    print(
        f"Minimum abstract similarity: "
        f"{result['minimum_abstract_score']:.1f}"
    )
    print()

    print("Pairwise abstract scores:")
    for pair in result["pair_scores"]:
        print(
            f"  {pair['eid_1']} vs {pair['eid_2']}: "
            f"{pair['abstract_score']:.1f}"
        )

    print()

    for _, row in group.iterrows():
        print(f"eID.     : {row['eid']}")
        print(f"Title    : {row['title']}")
        print(f"Abstract : {row['abstract']}")
        print("-" * 100)

    print()

In [ ]:
from itertools import combinations

import pandas as pd
from rapidfuzz import fuzz


FUZZY_THRESHOLD = 90


def prepare_text(value):
    if pd.isna(value):
        return ""

    return " ".join(str(value).lower().split())


same_title_dois = title_comparison.loc[
    title_comparison["unique_title_count"] == 1,
    "doi",
]

same_title_rows = (
    duplicate_df.loc[
        duplicate_df["doi"].isin(same_title_dois),
        ["doi", "eid", "title", "abstract"],
    ]
    .sort_values(["doi", "eid"])
    .copy()
)

comparison_results = []

for doi, group in same_title_rows.groupby("doi"):
    records = group.to_dict("records")
    pair_scores = []

    for record_1, record_2 in combinations(records, 2):
        abstract_score = fuzz.ratio(
            prepare_text(record_1["abstract"]),
            prepare_text(record_2["abstract"]),
        )

        pair_scores.append(
            {
                "eid_1": record_1["eid"],
                "eid_2": record_2["eid"],
                "abstract_score": abstract_score,
            }
        )

    minimum_abstract_score = min(
        pair["abstract_score"] for pair in pair_scores
    )

    comparison_results.append(
        {
            "doi": doi,
            "record_count": len(group),
            "minimum_abstract_score": minimum_abstract_score,
            "passed": minimum_abstract_score >= FUZZY_THRESHOLD,
            "pair_scores": pair_scores,
        }
    )

same_title_abstract_comparison = pd.DataFrame(comparison_results)

total_groups = len(same_title_abstract_comparison)
passed_groups = same_title_abstract_comparison["passed"].sum()
failed_groups = total_groups - passed_groups

print(f"Same-title DOI groups checked       : {total_groups:,}")
print(f"Groups passing abstract similarity  : {passed_groups:,}")
print(f"Groups failing abstract similarity  : {failed_groups:,}")

if total_groups:
    print(f"Pass rate                           : {passed_groups / total_groups:.2%}")

In [ ]:
failed_same_title_groups = (
    same_title_abstract_comparison.loc[
        ~same_title_abstract_comparison["passed"]
    ]
    .sort_values(
        ["minimum_abstract_score", "doi"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

for c, (_, result) in enumerate(
    failed_same_title_groups.iterrows(),
    start=1,
):
    doi = result["doi"]

    group = same_title_rows.loc[
        same_title_rows["doi"] == doi
    ]

    print("=" * 100)
    print(f"Number: {c}")
    print(f"DOI: {doi}")
    print(f"Number of records: {len(group)}")
    print(
        f"Minimum abstract similarity: "
        f"{result['minimum_abstract_score']:.1f}"
    )
    print()

    print("Pairwise abstract scores:")
    for pair in result["pair_scores"]:
        print(
            f"  {pair['eid_1']} vs {pair['eid_2']}: "
            f"{pair['abstract_score']:.1f}"
        )

    print()

    for _, row in group.iterrows():
        print(f"eID.     : {row['eid']}")
        print(f"Title    : {row['title']}")
        print(f"Abstract : {row['abstract']}")
        print("-" * 100)

    print()

In [ ]:
# Start from a separate copy.
df_cleaned = df.copy(deep=True)

# Remove records containing "Corrigendum" or "Proceedings"
pattern = r"corrigendum|proceedings"

mask = (
    df_cleaned["title"].str.contains(pattern, case=False, na=False)
    | df_cleaned["abstract"].str.contains(pattern, case=False, na=False)
)

print(f"Removing {mask.sum():,} records.")

df_cleaned = (
    df_cleaned.loc[~mask]
    .reset_index(drop=True)
)

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned.head()

In [ ]:
# Remove duplicate normalized DOIs, keeping the first occurrence.
df_cleaned = (
    df_cleaned
    .drop_duplicates(subset="doi", keep="first")
    .reset_index(drop=True)
)

In [ ]:
df_cleaned

In [ ]:
sql.list_tables("../data/scopus_elsevier.db")

In [ ]:
sql.get_table_schema("../data/scopus_elsevier.db", "papers_cleaned")

In [ ]:
# with sqlite3.connect("../data/scopus_elsevier.db") as conn:
#     cur = conn.cursor()

#     # Drop the table if it already exists
#     cur.execute("DROP TABLE IF EXISTS papers_dup")

#     # Create the new table
#     cur.execute("""
#         CREATE TABLE papers_dup (
#             eid TEXT PRIMARY KEY,
#             doi TEXT,
#             title TEXT,
#             journal TEXT,
#             publisher TEXT,
#             abstract TEXT
#         )
#     """)

#     # Insert the dataframe
#     df_cleaned.to_sql(
#         "papers_dup",
#         conn,
#         if_exists="append",
#         index=False,
#         chunksize=500,
#     )

# print(f"Inserted {len(df_cleaned):,} rows into papers_dup.")

#### Wiley

lesser duplicates here

In [ ]:
df = sql.load_table("../data/scopus_wiley.db", "papers_cleaned")

In [ ]:
df.head()

In [ ]:
def normalize_doi(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if not value:
        return pd.NA

    # Remove common DOI prefixes
    value = re.sub(
        r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Remove surrounding whitespace and common trailing punctuation
    value = value.strip().rstrip(".,;")

    return value if value else pd.NA

In [ ]:
df["doi"] = df["doi"].apply(normalize_doi)

In [ ]:
total_rows = len(df)

unique_dois = df["doi"].nunique(dropna=True)

duplicate_rows = total_rows - unique_dois

duplicate_dois = (
    df["doi"]
    .dropna()
    .value_counts()
    .loc[lambda x: x > 1]
)

print(f"Total rows           : {total_rows:,}")
print(f"Unique DOIs          : {unique_dois:,}")
print(f"Duplicate rows       : {duplicate_rows:,}")
print(f"Duplicate DOI values : {len(duplicate_dois):,}")

In [ ]:
dup_distribution = (
    duplicate_dois
    .value_counts()
    .sort_index()
    .rename_axis("times_appearing")
    .reset_index(name="number_of_dois")
)

print(dup_distribution)

In [ ]:
doi = duplicate_dois.index[0]

example = (
    df[df["doi"] == doi]
    .sort_values("eid")
)

example

In [ ]:
def normalize_title(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if not value:
        return pd.NA

    # Remove punctuation and normalize whitespace.
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value if value else pd.NA


df_check = df.copy()
df_check["title_normalized"] = df_check["title"].map(normalize_title)

# DOI counts
doi_counts = df_check["doi"].value_counts()

# Keep only rows whose DOI appears more than once
duplicate_df = df_check[
    df_check["doi"].isin(doi_counts[doi_counts > 1].index)
].copy()

# Count distinct normalized titles per duplicated DOI
title_comparison = (
    duplicate_df
    .groupby("doi")
    .agg(
        row_count=("doi", "size"),
        unique_title_count=("title_normalized", "nunique"),
        titles=("title", lambda x: list(pd.unique(x))),
        normalized_titles=(
            "title_normalized",
            lambda x: list(pd.unique(x.dropna())),
        ),
    )
    .reset_index()
)

same_title_count = (
    title_comparison["unique_title_count"] == 1
).sum()

different_title_count = (
    title_comparison["unique_title_count"] > 1
).sum()

print(
    f"Duplicate DOIs with the same normalized title    : "
    f"{same_title_count:,}"
)
print(
    f"Duplicate DOIs with different normalized titles : "
    f"{different_title_count:,}"
)
print(
    f"Total duplicated DOI groups                      : "
    f"{len(title_comparison):,}"
)

In [ ]:
different_titles = title_comparison[
    title_comparison["unique_title_count"] > 1
].copy()

different_titles.head()

In [ ]:
different_title_dois = title_comparison.loc[
    title_comparison["unique_title_count"] > 1,
    "doi",
]

different_title_rows = (
    duplicate_df.loc[
        duplicate_df["doi"].isin(different_title_dois),
        ["doi", "eid", "title", "abstract"],
    ]
    .sort_values(["doi", "eid"])
)

c = 0
for doi, group in different_title_rows.groupby("doi"):
    c+=1
    print("=" * 100)
    print(f"Number: {c}")
    print(f"DOI: {doi}")
    print(f"Number of records: {len(group)}")
    print()

    for _, row in group.iterrows():
        print(f"eID       : {row['eid']}")
        print(f"Title     : {row['title']}")
        print(f"Abstract  : {row['abstract']}")
        print("-" * 100)

    print()

In [ ]:
eids_to_remove = ["2-s2.0-0032210429", "2-s2.0-85050902869", "2-s2.0-84955082810", "2-s2.0-84955150369", "2-s2.0-84889788851",
                  "2-s2.0-84889858271", "2-s2.0-84889762882", "2-s2.0-84889767326"]

dois_to_remove = ["10.1002/9783527618224.ch5a", "10.1002/9783527618224.ch5b", "10.1002/9783527618224.ch5c", "10.1002/9783527618224.ch6",
                  "10.1002/9783527619306", "10.1002/9783527619313", "10.1002/9783527620777.ch2ce", "10.1002/9783527627240", "10.1002/9783527631094",
                  "10.1002/9783527631193", "10.1002/9783527631421", "10.1002/9783527631780", "10.1002/9783527631803", "10.1002/9783527631827",
                  "10.1002/9783527633197", "10.1002/9783527634408", "10.1002/9783527638482", "10.1002/9783527639953", "10.1002/9783527644582.ch1",
                  "10.1002/9783527645213", "10.1002/9783527674107", "10.1002/9783527689897", "10.1002/app.51667", "10.1002/jpln.201600453", "10.1002/pc.26459",
                  "10.1109/9780470545263.sect12", "10.1109/9780470545263.sect4"]

doi_change = {
    "2-s2.0-0347385155": "10.1002/jbm.a.10167"
}

In [ ]:
from wiley_tdm import TDMClient

tdm = TDMClient(api_token="4132c2b5-dd34-420c-96ab-c8360d231ce5")

In [ ]:
tdm.download_pdf("10.1002/jbm.a.10167")

In [ ]:
# Start from a separate copy.
df_cleaned = df.copy(deep=True)

# 1. Update DOI for specific eIDs.
mask = df_cleaned["eid"].isin(doi_change)

df_cleaned.loc[mask, "doi"] = (
    df_cleaned.loc[mask, "eid"]
    .map(doi_change)
)

# 2. Remove specified eIDs.
df_cleaned = df_cleaned.loc[
    ~df_cleaned["eid"].isin(eids_to_remove)
].copy()

# 3. Remove specified DOIs.
df_cleaned = df_cleaned.loc[
    ~df_cleaned["doi"].isin(dois_to_remove)
].copy()

# 4. Deduplicate by DOI, keeping the first occurrence.
df_cleaned = (
    df_cleaned
    .drop_duplicates(subset="doi", keep="first")
    .reset_index(drop=True)
)

In [ ]:
df_cleaned

In [ ]:
sql.get_table_schema("../data/scopus_wiley.db", "papers_cleaned")

In [ ]:
# with sqlite3.connect("../data/scopus_wiley.db") as conn:
#     cur = conn.cursor()

#     # Drop the table if it already exists
#     cur.execute("DROP TABLE IF EXISTS papers_dup")

#     # Create the new table
#     cur.execute("""
#         CREATE TABLE papers_dup (
#             eid TEXT PRIMARY KEY,
#             doi TEXT,
#             title TEXT,
#             journal TEXT,
#             publisher TEXT,
#             abstract TEXT
#         )
#     """)

#     # Insert the dataframe
#     df_cleaned.to_sql(
#         "papers_dup",
#         conn,
#         if_exists="append",
#         index=False,
#         chunksize=500,
#     )

# print(f"Inserted {len(df_cleaned):,} rows into papers_dup.")

#### PMC

In [ ]:
df = sql.load_table("../data/europepmc.db", "papers_cleaned")

In [ ]:
df.head()

In [ ]:
def normalize_doi(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if not value:
        return pd.NA

    # Remove common DOI prefixes
    value = re.sub(
        r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Remove surrounding whitespace and common trailing punctuation
    value = value.strip().rstrip(".,;")

    return value if value else pd.NA

In [ ]:
df["doi"] = df["doi"].apply(normalize_doi)

In [ ]:
total_rows = len(df)

unique_dois = df["doi"].nunique(dropna=True)

duplicate_rows = total_rows - unique_dois

duplicate_dois = (
    df["doi"]
    .dropna()
    .value_counts()
    .loc[lambda x: x > 1]
)

print(f"Total rows           : {total_rows:,}")
print(f"Unique DOIs          : {unique_dois:,}")
print(f"Duplicate rows       : {duplicate_rows:,}")
print(f"Duplicate DOI values : {len(duplicate_dois):,}")

In [ ]:
dup_distribution = (
    duplicate_dois
    .value_counts()
    .sort_index()
    .rename_axis("times_appearing")
    .reset_index(name="number_of_dois")
)

print(dup_distribution)

In [ ]:
doi = duplicate_dois.index[0]

example = (
    df[df["doi"] == doi]
    .sort_values("pmcid")
)

example

In [ ]:
def normalize_title(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if not value:
        return pd.NA

    # Remove punctuation and normalize whitespace.
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value if value else pd.NA


df_check = df.copy()
df_check["title_normalized"] = df_check["title"].map(normalize_title)

# DOI counts
doi_counts = df_check["doi"].value_counts()

# Keep only rows whose DOI appears more than once
duplicate_df = df_check[
    df_check["doi"].isin(doi_counts[doi_counts > 1].index)
].copy()

# Count distinct normalized titles per duplicated DOI
title_comparison = (
    duplicate_df
    .groupby("doi")
    .agg(
        row_count=("doi", "size"),
        unique_title_count=("title_normalized", "nunique"),
        titles=("title", lambda x: list(pd.unique(x))),
        normalized_titles=(
            "title_normalized",
            lambda x: list(pd.unique(x.dropna())),
        ),
    )
    .reset_index()
)

same_title_count = (
    title_comparison["unique_title_count"] == 1
).sum()

different_title_count = (
    title_comparison["unique_title_count"] > 1
).sum()

print(
    f"Duplicate DOIs with the same normalized title    : "
    f"{same_title_count:,}"
)
print(
    f"Duplicate DOIs with different normalized titles : "
    f"{different_title_count:,}"
)
print(
    f"Total duplicated DOI groups                      : "
    f"{len(title_comparison):,}"
)

In [ ]:
different_title_dois = title_comparison.loc[
    title_comparison["unique_title_count"] > 1,
    "doi",
]

different_title_rows = (
    duplicate_df.loc[
        duplicate_df["doi"].isin(different_title_dois),
        ["doi", "pmcid", "title", "abstract"],
    ]
    .sort_values(["doi", "pmcid"])
)

c = 0
for doi, group in different_title_rows.groupby("doi"):
    c+=1
    print("=" * 100)
    print(f"Number: {c}")
    print(f"DOI: {doi}")
    print(f"Number of records: {len(group)}")
    print()

    for _, row in group.iterrows():
        print(f"PMCID     : {row['pmcid']}")
        print(f"Title     : {row['title']}")
        print(f"Abstract  : {row['abstract']}")
        print("-" * 100)

    print()

In [ ]:
different_title_dois = title_comparison.loc[
    title_comparison["unique_title_count"] == 1,
    "doi",
]

different_title_rows = (
    duplicate_df.loc[
        duplicate_df["doi"].isin(different_title_dois),
        ["doi", "pmcid", "title", "abstract"],
    ]
    .sort_values(["doi", "pmcid"])
)

c = 0
for doi, group in different_title_rows.groupby("doi"):
    c+=1
    print("=" * 100)
    print(f"Number: {c}")
    print(f"DOI: {doi}")
    print(f"Number of records: {len(group)}")
    print()

    for _, row in group.iterrows():
        print(f"PMCID     : {row['pmcid']}")
        print(f"Title     : {row['title']}")
        print(f"Abstract  : {row['abstract']}")
        print("-" * 100)

    print()

In [ ]:
doi_change = {
    "PMC2110041": "10.1083/jcb.77.2.551"
}

In [ ]:
# Start from a separate copy.
df_cleaned = df.copy(deep=True)

# 1. Update DOI for specific eIDs.
mask = df_cleaned["pmcid"].isin(doi_change)

df_cleaned.loc[mask, "doi"] = (
    df_cleaned.loc[mask, "pmcid"]
    .map(doi_change)
)

# 4. Deduplicate by DOI, keeping the first occurrence.
df_cleaned = (
    df_cleaned
    .drop_duplicates(subset="doi", keep="first")
    .reset_index(drop=True)
)

In [ ]:
df_cleaned

In [ ]:
sql.get_table_schema("../data/europepmc.db", "papers_cleaned")

In [ ]:
# with sqlite3.connect("../data/europepmc.db") as conn:
#     cur = conn.cursor()

#     # Drop the table if it already exists
#     cur.execute("DROP TABLE IF EXISTS papers_dup")

#     # Create the new table
#     cur.execute("""
#         CREATE TABLE papers_dup (
#             pmcid TEXT PRIMARY KEY,
#             pmid TEXT,
#             doi TEXT,
#             title TEXT,
#             journal TEXT,
#             abstract TEXT
#         )
#     """)

#     # Insert the dataframe
#     df_cleaned.to_sql(
#         "papers_dup",
#         conn,
#         if_exists="append",
#         index=False,
#         chunksize=500,
#     )

# print(f"Inserted {len(df_cleaned):,} rows into papers_dup.")

#### Arxiv

arxiv is more of a special case. we need to check

In [ ]:
sql.list_tables("../data/arxiv.db")

In [ ]:
sql.get_table_schema("../data/arxiv.db", "papers_cleaned")

In [ ]:
df = sql.load_table("../data/arxiv.db", "papers_cleaned")

In [ ]:
df.head()

In [ ]:
len(df[df["doi"].notna()]), len(df)

In [ ]:
df["journal_ref"].notna().sum()

In [ ]:
sources = [
    {
        "source": "wiley",
        "db_path": "../data/scopus_wiley.db",
        "table": "papers_dup",
    },
    {
        "source": "springer_nature",
        "db_path": "../data/scopus_springer_nature.db",
        "table": "papers_dup",
    },
    {
        "source": "elsevier",
        "db_path": "../data/scopus_elsevier.db",
        "table": "papers_dup",
    },
    {
        "source": "s2orc",
        "db_path": "../data/s2orc.db",
        "table": "papers_dup",
    },
    {
        "source": "europepmc",
        "db_path": "../data/europepmc.db",
        "table": "papers_dup",
    },
]

frames = []

for source in sources:
    if source["source"] == "arxiv":
        continue

    df = sql.load_table(
        source["db_path"],
        source["table"],
    ).copy()

    df["source"] = source["source"]

    frames.append(
        df[["doi", "title", "abstract", "source"]]
    )

df_all = pd.concat(frames, ignore_index=True)

based on below we have a scheme to create arxiv dois and populate (Not sure if its general)

In [ ]:
import requests

DATACITE_API = "https://api.datacite.org/dois"
with requests.Session() as session:
        session.headers.update({
            "User-Agent": (
                "polymer-corpus-deduplication/1.0 "
                "(contact: your-email@example.com)"
            )
        })
        arxiv_doi = f"10.48550/arxiv.hep-th/{9108014}"
        url = f"{DATACITE_API}/{arxiv_doi}"

        response = session.get(url, timeout=60)

In [ ]:
response.json()

In [ ]:
duplicate_dois = (
    df["doi"]
    .value_counts()
    .loc[lambda s: s > 1]
    .index
)

duplicate_df = (
    df[df["doi"].isin(duplicate_dois)]
    .sort_values(["doi", "published"])
)

print(f"Duplicate DOI groups: {len(duplicate_dois):,}")

import textwrap

for doi, group in duplicate_df.groupby("doi"):
    print("=" * 120)
    print(f"DOI: {doi}")
    print(f"Number of records: {len(group)}")
    print()

    for _, row in group.sort_values("published").iterrows():
        print(f"arXiv ID : {row['arxiv_id']}")
        print(f"Published: {row['published']}")
        print(
            textwrap.fill(
                row["title"],
                width=100,
                initial_indent="Title    : ",
                subsequent_indent="           ",
            )
        )
        print("-" * 120)

    print()

In [ ]:
print(f"Rows before deduplication : {len(df):,}")
df["published"] = pd.to_datetime(df["published"])

has_doi = df["doi"].notna()

df_with_doi = (
    df.loc[has_doi]
    .sort_values("published")
    .drop_duplicates(subset="doi", keep="last")
)

df_without_doi = df.loc[~has_doi]

df = (
    pd.concat(
        [df_with_doi, df_without_doi],
        ignore_index=True,
    )
    .sort_values("published")
    .reset_index(drop=True)
)

print(f"Rows after deduplication : {len(df):,}")
print(f"Unique non-null DOIs     : {df['doi'].nunique():,}")
print(f"Rows without DOI         : {df['doi'].isna().sum():,}")

In [ ]:
def normalize_doi(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if not value:
        return pd.NA

    # Remove common DOI prefixes
    value = re.sub(
        r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Remove surrounding whitespace and common trailing punctuation
    value = value.strip().rstrip(".,;")

    return value if value else pd.NA

In [ ]:
df["doi"] = df["doi"].apply(normalize_doi)

In [ ]:
sql.get_table_schema("../data/arxiv.db", "papers_cleaned")

In [ ]:
# # save to sql

# with sqlite3.connect("../data/arxiv.db") as conn:
#     cur = conn.cursor()

#     # Drop the table if it already exists.
#     cur.execute("DROP TABLE IF EXISTS papers_dup")

#     # Create the new table.
#     cur.execute("""
#         CREATE TABLE papers_dup (
#             arxiv_id TEXT PRIMARY KEY,
#             version INTEGER,
#             arxiv_url TEXT,
#             doi TEXT,
#             title TEXT,
#             abstract TEXT,
#             authors TEXT,
#             affiliation TEXT,
#             primary_category TEXT,
#             categories TEXT,
#             published TEXT,
#             updated TEXT,
#             journal_ref TEXT,
#             comment TEXT,
#             pdf_url TEXT
#         )
#     """)

#     # Insert the cleaned DataFrame.
#     df.to_sql(
#         "papers_dup",
#         conn,
#         if_exists="append",
#         index=False,
#         chunksize=500,
#     )

# print(f"Inserted {len(df):,} rows into papers_dup.")

arxiv is kind of difficult to manage so we treat it differently for now. 

Trying to fill dois of arxiv is not really worth it

#### Cross duplication analysis

In [139]:
sources = [
    {
        "source": "wiley",
        "db_path": "../data/scopus_wiley.db",
        "table": "papers_dup",
    },
    {
        "source": "springer_nature",
        "db_path": "../data/scopus_springer_nature.db",
        "table": "papers_dup",
    },
    {
        "source": "elsevier",
        "db_path": "../data/scopus_elsevier.db",
        "table": "papers_dup",
    },
    {
        "source": "s2orc",
        "db_path": "../data/s2orc.db",
        "table": "papers_dup",
    },
    {
        "source": "europepmc",
        "db_path": "../data/europepmc.db",
        "table": "papers_dup",
    },
    {
        "source": "arxiv",
        "db_path": "../data/arxiv.db",
        "table": "papers_dup",
    },
]

In [140]:
frames = []

for source in sources:
    # if source["source"] == "arxiv":
    #     continue

    print(f"taking from source: {source['source']}")
    df = sql.load_table(
        source["db_path"],
        source["table"],
    ).copy()

    df["source"] = source["source"]

    frames.append(
        df[["doi", "title", "abstract", "source"]]
    )

df_all = pd.concat(frames, ignore_index=True)

taking from source: wiley
taking from source: springer_nature
taking from source: elsevier
taking from source: s2orc
taking from source: europepmc
taking from source: arxiv


In [141]:
total_rows = len(df_all)
rows_with_doi = df_all["doi"].notna().sum()
rows_without_doi = df_all["doi"].isna().sum()
unique_dois = df_all.loc[df_all["doi"].notna(), "doi"].nunique()

print(f"Total rows                : {total_rows:,}")
print(f"Rows containing DOI       : {rows_with_doi:,}")
print(f"Rows without DOI (arXiv)  : {rows_without_doi:,}")
print(f"Unique non-null DOIs      : {unique_dois:,}")

Total rows                : 1,347,478
Rows containing DOI       : 1,341,749
Rows without DOI (arXiv)  : 5,729
Unique non-null DOIs      : 1,096,181


In [142]:
doi_summary = (
    df_all[df_all["doi"].notna()]
    .groupby("doi")
    .agg(
        record_count=("doi", "size"),
        source_count=("source", "nunique"),
        sources=("source", lambda x: sorted(set(x))),
    )
    .reset_index()
)

source_count_stats = (
    doi_summary["source_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_sources")
    .reset_index(name="number_of_dois")
)

print(source_count_stats)

   number_of_sources  number_of_dois
0                  1          868396
1                  2          210041
2                  3           17705
3                  4              39


In [143]:
doi_summary["source_combination"] = (
    doi_summary["sources"]
    .apply(tuple)
)

combination_stats = (
    doi_summary["source_combination"]
    .value_counts()
)

print(combination_stats)

source_combination
(elsevier,)                                   481378
(europepmc, s2orc)                            183099
(wiley,)                                      172147
(s2orc,)                                      129459
(europepmc,)                                   71734
(springer_nature,)                             13213
(europepmc, s2orc, wiley)                       7823
(elsevier, europepmc, s2orc)                    7702
(arxiv, s2orc)                                  6833
(elsevier, s2orc)                               5699
(s2orc, wiley)                                  4899
(elsevier, europepmc)                           4236
(europepmc, wiley)                              4206
(europepmc, s2orc, springer_nature)              813
(s2orc, springer_nature)                         660
(arxiv, europepmc, s2orc)                        593
(arxiv, elsevier, s2orc)                         587
(arxiv,)                                         465
(europepmc, springer_nature

In [144]:
doi_summary

,doi,record_count,source_count,sources,source_combination
0,0.1007/s10652-005-0611-3,1,1,[arxiv],"(arxiv,)"
1,0.1016/j.triboint.2023.109203,1,1,[arxiv],"(arxiv,)"
2,10.0001/223,1,1,[s2orc],"(s2orc,)"
3,10.02/9783527822850,1,1,[wiley],"(wiley,)"
4,10.1001/2013.jamaneurol.537,1,1,[europepmc],"(europepmc,)"
...,...,...,...,...,...
1096176,10.9790/5736-1006020825,1,1,[s2orc],"(s2orc,)"
1096177,10.9790/5736-1007010710,1,1,[s2orc],"(s2orc,)"
1096178,10.9790/5736-7910921,1,1,[s2orc],"(s2orc,)"
1096179,10.9790/6737-0140105,1,1,[s2orc],"(s2orc,)"


In [145]:
publisher_sources = {
    "elsevier",
    "wiley",
    "springer_nature",
}

# Preserve doi_summary unchanged.
publisher_overlap = doi_summary.copy()

# Identify which of the three publisher sources contain each DOI.
publisher_overlap["publisher_sources"] = (
    publisher_overlap["sources"]
    .apply(
        lambda sources: sorted(
            set(sources) & publisher_sources
        )
    )
)

publisher_overlap["publisher_source_count"] = (
    publisher_overlap["publisher_sources"].str.len()
)

# Keep DOIs found in at least two of the three publishers.
publisher_overlap = (
    publisher_overlap.loc[
        publisher_overlap["publisher_source_count"] >= 2
    ]
    .sort_values(
        ["publisher_source_count", "doi"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print(
    f"DOIs found in at least two of Elsevier, Wiley, "
    f"and Springer Nature: {len(publisher_overlap):,}"
)

publisher_combination_stats = (
    publisher_overlap["publisher_sources"]
    .apply(tuple)
    .value_counts()
    .rename_axis("publisher_combination")
    .reset_index(name="number_of_dois")
)

print(publisher_combination_stats)

DOIs found in at least two of Elsevier, Wiley, and Springer Nature: 156
  publisher_combination  number_of_dois
0     (elsevier, wiley)             156


In [136]:
rows = df_all[
    df_all["doi"].isin(publisher_overlap["doi"])
].sort_values(["doi", "source"])

rows

,doi,title,abstract,source
584476,10.1002/2211-5463.12002,A specific role for PRND in goat foetal Leydig...,Three genes of the prion protein gene family a...,elsevier
140203,10.1002/2211-5463.12002,A specific role for PRND in goat foetal Leydig...,Three genes of the prion protein gene family a...,wiley
578025,10.1002/2211-5463.12027,miRNA expression profiling of Epstein-Barr vir...,The aim of this work was to establish the micr...,elsevier
138297,10.1002/2211-5463.12027,miRNA expression profiling of Epstein-Barr vir...,The aim of this work was to establish the micr...,wiley
581222,10.1002/2211-5463.12030,Clinical significance of long noncoding RNA SP...,Long noncoding RNA SPRY4-IT1 has been reported...,elsevier
...,...,...,...,...
185659,10.1016/s0014-5793(96)01189-1,Characterisation of a human homologue of a yea...,Four exons of a human homologue of a yeast cel...,wiley
693555,10.1016/s0014-5793(97)00820-x,Molecular and functional characterization of a...,5-Hydroxytryptamine (5-HT) has been shown to e...,elsevier
182840,10.1016/s0014-5793(97)00820-x,Molecular and functional characterization of a...,5-Hydroxytryptamine (5-HT) has been shown to e...,wiley
688537,10.1016/s0014-5793(98)00878-3,A different isoform of the transport protein m...,There are differences in the kinetic propertie...,elsevier


no need to delete them. just put wiley over elsevier and done! and move to download!

In [146]:
duplicate_df = df_all[
    df_all["doi"].isin(
        doi_summary.loc[
            doi_summary["source_count"] > 1,
            "doi",
        ]
    )
].copy()

In [147]:
duplicate_df

,doi,title,abstract,source
2,10.1002/rcm.70102,Liquid Atmospheric Pressure MALDI (LAP-MALDI) ...,Liquid atmospheric pressure matrix‐assisted la...,wiley
4,10.1002/rcm.70073,Resolving Isomeric Structures in Natural Compl...,Recent advances in ion mobility mass spectrome...,wiley
5,10.1002/ijc.70514,Clinical and Radiometabolic Correlatives of dd...,Circulating tumor DNA (ctDNA) as a liquid biop...,wiley
6,10.1002/jsfa.70632,Comparative analysis of microRNAs in bovine co...,MicroRNAs (miRNAs) are short non‐coding RNAs t...,wiley
61,10.1111/jne.70215,Ovarian hormones and high-fat diet duration di...,Excessive intake of saturated fats triggers in...,wiley
...,...,...,...,...
1347386,10.1016/j.physleta.2026.131807,B-Spline for Self-Consistent Field Theory with...,Polymer self-consistent field theory (SCFT) ha...,arxiv
1347404,10.1002/prop.70097,Reflection-Positive Construction of a Four-Dim...,In the Euclidean view one must first require t...,arxiv
1347417,10.1016/j.jprocont.2026.103738,Real-time nonlinear model predictive control f...,Controlling batch polymerization is challengin...,arxiv
1347453,10.1088/1361-648x/ab2ac2,"The interplay of interfaces, supramolecular as...","Organic semiconductors, which include a divers...",arxiv


In [148]:
import html

In [149]:
# Spelled-out Greek helps titles like "α-synuclein" match "alpha-synuclein"
GREEK_MAP = str.maketrans({
    "α": " alpha ", "β": " beta ", "γ": " gamma ", "δ": " delta ",
    "ε": " epsilon ", "θ": " theta ", "κ": " kappa ", "λ": " lambda ",
    "μ": " mu ", "π": " pi ", "σ": " sigma ", "φ": " phi ", "ω": " omega ",
})
 
 
def normalize_title(value):
    """Normalize a paper title for duplicate detection."""
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    if not value:
        return pd.NA
 
    value = html.unescape(value)              # &amp; -> &, &#8217; -> ’
    value = re.sub(r"<[^>]+>", " ", value)     # strip HTML/XML tags, keep spacing
    value = html.unescape(value)               # catch entities revealed under tags
 
    value = unicodedata.normalize("NFKC", value)  # full-width chars, ligatures,
                                                   # AND sub/superscript digits -> plain digits
    value = value.lower()
    value = value.translate(GREEK_MAP)             # α -> " alpha "
 
    value = re.sub(r"[‐-‒–—−]", "-", value)    # dash variants -> "-"
    value = re.sub(r"[‘’´`]", "'", value)      # quote variants -> "'"
 
    value = re.sub(r"[^\w\s]", " ", value)     # remaining punctuation -> space
    value = re.sub(r"\s+", " ", value).strip() # collapse whitespace
 
    return value if value else pd.NA

In [150]:
duplicate_df["title_normalized"] = (
    duplicate_df["title"]
    .map(normalize_title)
)

title_stats = (
    duplicate_df
    .groupby("doi")
    .agg(
        source_count=("source", "nunique"),
        unique_titles=("title_normalized", "nunique"),
    )
    .reset_index()
)

print(f"Duplicate DOIs: {len(title_stats):,}")
print(f"Same normalized title: {(title_stats['unique_titles'] == 1).sum():,}")
print(f"Different normalized title: {(title_stats['unique_titles'] > 1).sum():,}")

Duplicate DOIs: 227,785
Same normalized title: 218,623
Different normalized title: 9,162


most of the title looks the same some sapce and other issues afeect the normalization

In [151]:
from rapidfuzz import fuzz
import pandas as pd


def compare_titles(
    title_1,
    title_2,
    ratio_threshold=90,
    partial_threshold=90,
):
    title_1 = normalize_title(title_1)
    title_2 = normalize_title(title_2)

    if pd.isna(title_1) or pd.isna(title_2):
        return {
            "ratio": 0.0,
            "partial_ratio": 0.0,
            "passed": False,
        }

    ratio = fuzz.ratio(title_1, title_2)
    partial_ratio = fuzz.partial_ratio(title_1, title_2)

    passed = (
        ratio >= ratio_threshold
        or partial_ratio >= partial_threshold
    )

    return {
        "ratio": ratio,
        "partial_ratio": partial_ratio,
        "passed": passed,
    }

In [152]:
comparison_records = []

for doi, group in duplicate_df.groupby("doi"):
    records = group[["source", "title"]].to_dict("records")
    pair_results = []

    for record_1, record_2 in combinations(records, 2):
        result = compare_titles(
            record_1["title"],
            record_2["title"],
        )

        pair_results.append(
            {
                "source_1": record_1["source"],
                "source_2": record_2["source"],
                **result,
            }
        )

    # Pass only when every pair in the DOI group passes.
    all_titles_match = all(
        pair["passed"]
        for pair in pair_results
    )

    comparison_records.append(
        {
            "doi": doi,
            "source_count": group["source"].nunique(),
            "record_count": len(group),
            "minimum_ratio": min(
                pair["ratio"]
                for pair in pair_results
            ),
            "minimum_partial_ratio": min(
                pair["partial_ratio"]
                for pair in pair_results
            ),
            "all_titles_match": all_titles_match,
            "pair_results": pair_results,
        }
    )

title_stats = pd.DataFrame(comparison_records)

total_duplicate_dois = len(title_stats)
matching_title_dois = title_stats["all_titles_match"].sum()
different_title_dois = total_duplicate_dois - matching_title_dois

print(f"Duplicate DOI groups            : {total_duplicate_dois:,}")
print(f"Passed fuzzy title comparison   : {matching_title_dois:,}")
print(f"Failed fuzzy title comparison   : {different_title_dois:,}")

if total_duplicate_dois:
    print(
        f"Fuzzy title pass rate           : "
        f"{matching_title_dois / total_duplicate_dois:.2%}"
    )

Duplicate DOI groups            : 227,785
Passed fuzzy title comparison   : 227,390
Failed fuzzy title comparison   : 395
Fuzzy title pass rate           : 99.83%


In [153]:
failed_title_groups = (
    title_stats.loc[
        ~title_stats["all_titles_match"]
    ]
    .sort_values(
        ["minimum_partial_ratio", "minimum_ratio"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

N = 50  # print the first 50 failures

for i, (_, result) in enumerate(
    failed_title_groups.head(N).iterrows(),
    start=1,
):
    doi = result["doi"]

    group = (
        duplicate_df.loc[
            duplicate_df["doi"] == doi,
            ["source", "title_normalized"]
        ]
        .sort_values("source")
    )

    print("=" * 120)
    print(f"#{i}")
    print(f"DOI: {doi}")
    print(
        f"Minimum ratio: {result['minimum_ratio']:.1f} | "
        f"Minimum partial ratio: {result['minimum_partial_ratio']:.1f}"
    )
    print()

    print("Pairwise scores:")
    for pair in result["pair_results"]:
        print(
            f"  {pair['source_1']:18s} ↔ {pair['source_2']:18s} "
            f"ratio={pair['ratio']:.1f} "
            f"partial={pair['partial_ratio']:.1f} "
        )

    print()

    for _, row in group.iterrows():
        print(f"Source : {row['source']}")
        print(
            textwrap.fill(
                row["title_normalized"],
                width=100,
                initial_indent="Title  : ",
                subsequent_indent="         ",
            )
        )
        print("-" * 120)

    print()

#1
DOI: 10.1073/pnas.1707251114
Minimum ratio: 35.2 | Minimum partial ratio: 38.0

Pairwise scores:
  europepmc          ↔ arxiv              ratio=35.2 partial=38.0 

Source : arxiv
Title  : direct test of supercooled liquid scaling relations
------------------------------------------------------------------------------------------------------------------------
Source : europepmc
Title  : toward broadband mechanical spectroscopy
------------------------------------------------------------------------------------------------------------------------

#2
DOI: 10.1140/epje/s10189-024-00461-4
Minimum ratio: 36.4 | Minimum partial ratio: 39.7

Pairwise scores:
  s2orc              ↔ arxiv              ratio=36.4 partial=39.7 

Source : arxiv
Title  : a detailed field theory for rna like molecules with periodic base sequence
------------------------------------------------------------------------------------------------------------------------
Source : s2orc
Title  : universality class of in

In [138]:
# doi_remove = {
#     "s2orc": ["10.3779/j.issn.1009-3419.2021.102.31", "10.1016/j.jss.2009.06.006", "10.1101/gad.178079.111", "10.1038/s41589-022-01220-2",
#               "10.1186/s12864-020-07014-x", "10.1103/physrevd.99.126013", "10.1097/pec.0000000000003274", "10.22074/cellj.2022.7763",
#               "10.3390/s7123027", "10.1038/s41551-021-00754-5", "10.1021/jacs.5b03482"],
#     "europepmc": ["10.3779/j.issn.1009-3419.2021.102.31", "10.1016/j.jss.2009.06.006", "10.1186/s12864-020-07014-x", "10.22074/cellj.2022.7763",
#                   "10.3390/s7123027"]
# }

# for source in sources:
#     source_name = source["source"]

#     if source_name not in doi_remove:
#         continue

#     db_path = source["db_path"]
#     table = source["table"]

#     with sqlite3.connect(db_path) as conn:
#         placeholders = ",".join("?" * len(doi_remove[source_name]))

#         conn.execute(
#             f"""
#             DELETE FROM {table}
#             WHERE doi IN ({placeholders})
#             """,
#             doi_remove[source_name],
#         )

#         conn.commit()

#         print(
#             f"{source_name}: removed {conn.total_changes} rows"
#         )

s2orc: removed 11 rows
europepmc: removed 5 rows


#### Some Checks

In [154]:
sources = [
    {
        "source": "wiley",
        "db_path": "../data/scopus_wiley.db",
        "table": "papers_dup",
    },
    {
        "source": "springer_nature",
        "db_path": "../data/scopus_springer_nature.db",
        "table": "papers_dup",
    },
    {
        "source": "elsevier",
        "db_path": "../data/scopus_elsevier.db",
        "table": "papers_dup",
    },
    {
        "source": "s2orc",
        "db_path": "../data/s2orc.db",
        "table": "papers_dup",
    },
    {
        "source": "europepmc",
        "db_path": "../data/europepmc.db",
        "table": "papers_dup",
    },
    {
        "source": "arxiv",
        "db_path": "../data/arxiv.db",
        "table": "papers_dup",
    },
]

In [155]:
frames = []

for source in sources:
    # if source["source"] == "arxiv":
    #     continue

    print(f"taking from source: {source['source']}")
    df = sql.load_table(
        source["db_path"],
        source["table"],
    ).copy()

    df["source"] = source["source"]

    frames.append(
        df[["doi", "title", "abstract", "source"]]
    )

df_all = pd.concat(frames, ignore_index=True)

taking from source: wiley
taking from source: springer_nature
taking from source: elsevier
taking from source: s2orc
taking from source: europepmc
taking from source: arxiv


In [156]:
df_all.head()

,doi,title,abstract,source
0,10.1002/9781394346998.ch2,Norovirus: Its Role in Gastroenteritis Outbreaks,Noroviruses are a heterogeneous group of genet...,wiley
1,10.1002/ccs3.70090,Plasma exosomal miR-339-3p promotes myocardial...,To explore the mechanism by which plasma exoso...,wiley
2,10.1002/rcm.70102,Liquid Atmospheric Pressure MALDI (LAP-MALDI) ...,Liquid atmospheric pressure matrix‐assisted la...,wiley
3,10.1002/jsfa.70703,Beyond antimicrobial efficacy: Complex interac...,The gut microbiome is increasingly recognized ...,wiley
4,10.1002/rcm.70073,Resolving Isomeric Structures in Natural Compl...,Recent advances in ion mobility mass spectrome...,wiley


In [161]:
total_rows = len(df_all)
rows_with_doi = df_all["doi"].notna().sum()
rows_without_doi = df_all["doi"].isna().sum()
unique_dois = df_all.loc[df_all["doi"].notna(), "doi"].nunique()
total_unique = unique_dois + rows_without_doi

print(f"Total rows                : {total_rows:,}")
print(f"Rows containing DOI       : {rows_with_doi:,}")
print(f"Rows without DOI (arXiv)  : {rows_without_doi:,}")
print(f"Unique non-null DOIs      : {unique_dois:,}")
print(f"total unique entries      : {total_unique:,}")

Total rows                : 1,347,478
Rows containing DOI       : 1,341,749
Rows without DOI (arXiv)  : 5,729
Unique non-null DOIs      : 1,096,181
total unique entries      : 1,101,910


In [158]:
doi_summary = (
    df_all[df_all["doi"].notna()]
    .groupby("doi")
    .agg(
        record_count=("doi", "size"),
        source_count=("source", "nunique"),
        sources=("source", lambda x: sorted(set(x))),
    )
    .reset_index()
)

source_count_stats = (
    doi_summary["source_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_sources")
    .reset_index(name="number_of_dois")
)

print(source_count_stats)

   number_of_sources  number_of_dois
0                  1          868396
1                  2          210041
2                  3           17705
3                  4              39


In [160]:
print(f"Springer nature papers: {(df_all['source'] == 'springer_nature').sum()}")
print(f"Wiley papers: {(df_all['source'] == 'wiley').sum()}")
print(f"Elsevier papers: {(df_all['source'] == 'elsevier').sum()}")
print(f"EuropePMC papers: {(df_all['source'] == 'europepmc').sum()}")
print(f"S2ORC papers: {(df_all['source'] == 's2orc').sum()}")
print(f"arXiv papers: {(df_all['source'] == 'arxiv').sum()}")

Springer nature papers: 14913
Wiley papers: 189418
Elsevier papers: 499800
EuropePMC papers: 280477
S2ORC papers: 348389
arXiv papers: 14481


#### Handling wierd cases

In [ ]:
# updates = [
#     ("10.1007/s10652-005-0611-3", "0404010"),
#     ("10.1016/j.triboint.2023.109203", "2403.08373"),
# ]

# with sqlite3.connect("../data/arxiv.db") as conn:
#     conn.executemany(
#         """
#         UPDATE papers_dup
#         SET doi = ?
#         WHERE arxiv_id = ?
#         """,
#         updates,
#     )

#     conn.commit()

#     print(f"Updated {conn.total_changes} row(s).")

Updated 2 row(s).


In [164]:
with sqlite3.connect("../data/arxiv.db") as conn:
    df = pd.read_sql_query(
        """
        SELECT
            arxiv_id,
            doi,
            title,
            abstract
        FROM papers_dup
        WHERE arxiv_id IN (?, ?)
        """,
        conn,
        params=("0404010", "2403.08373"),
    )

df

,arxiv_id,doi,title,abstract
0,0404010,10.1007/s10652-005-0611-3,Simple analytical model for entire turbulent b...,We discuss a simple analytical model of the tu...
1,2403.08373,10.1016/j.triboint.2023.109203,Towards the superlubricity of polymer-steel in...,Frictional losses are responsible for signific...


In [ ]:
# import sqlite3

# with sqlite3.connect("../data/s2orc.db") as conn:
#     conn.execute(
#         """
#         UPDATE papers_dup
#         SET doi = ?
#         WHERE doi = ?
#         """,
#         (
#             "10.12816/0014927",
#             "10.0001/223",
#         ),
#     )

#     conn.commit()

#     print(f"Updated {conn.total_changes} row(s).")

Updated 1 row(s).


In [174]:
with sqlite3.connect("../data/s2orc.db") as conn:
    df = pd.read_sql_query(
        """
        SELECT *
        FROM papers_dup
        WHERE doi = ?
        """,
        conn,
        params=("10.12816/0014927",),
    )

df

,corpusid,doi,title,authors,abstract
0,137897935,10.12816/0014927,The effect of different finishing and polishin...,"[""M. Abdurazaq"", ""A. Al-Khafaji""]",Adequate finishing and polishing of resin comp...


In [171]:
df["title"].loc[0]

'The effect of different finishing and polishing systems on surface roughness of new low polymerized composite materials (An in vitro study)'